# 01 - Daten: VisDrone-DET 2019 vorbereiten

Download, COCO-Konvertierung, SAHI-Slicing (640x640 @ 20% Overlap),
Statistik und auto-generierter Report nach `docs/data_report.md`.

In [ ]:
# Zelle 1 - Setup: Drive + Repo + Dependencies
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess

REPO = '/content/ba-mamba-neck'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', 'https://github.com/raphaelkach/ba-mamba-neck.git', REPO],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, REPO)

!pip install -q mmengine pycocotools sahi matplotlib Pillow


In [ ]:
# Zelle 2 - VisDrone herunterladen
import os
from pathlib import Path

!pip install -q gdown

RAW = Path('/content/visdrone/raw')
RAW.mkdir(parents=True, exist_ok=True)

SPLITS = {
    'train': '1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn',
    'val':   '1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59',
}

for split, file_id in SPLITS.items():
    zip_name = f'VisDrone2019-DET-{split}.zip'
    dst = RAW / zip_name
    extract_dir = RAW / f'VisDrone2019-DET-{split}'

    if extract_dir.exists() and any(extract_dir.iterdir()):
        print(f'{split} bereits entpackt — skip')
        continue

    print(f'{split}: Download...')
    !gdown --fuzzy "https://drive.google.com/file/d/{file_id}/view" -O "{dst}"

    assert dst.exists() and dst.stat().st_size > 1_000_000, (
        f'Download {split} fehlgeschlagen. Manuell herunterladen von '
        f'https://github.com/VisDrone/VisDrone-Dataset und nach '
        f'{RAW} kopieren.'
    )

    print(f'{split}: Entpacken...')
    !cd "{RAW}" && unzip -q -o "{zip_name}"

print("Done — Train + Val bereit")

In [ ]:
# Zelle 3 - VisDrone konvertieren + SAHI-Slicing
!python data/prepare.py --output /content/visdrone

In [ ]:
# Zelle 4 - 5 Beispielbilder aus dem Val-Split mit Bounding Boxes
import json, random
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

ROOT = Path('/content/visdrone')
VAL_JSON = ROOT / 'annotations' / 'val_unsliced.json'
VAL_IMAGES = ROOT / 'raw' / 'VisDrone2019-DET-val' / 'images'

coco = json.loads(VAL_JSON.read_text())
id_to_name = {c['id']: c['name'] for c in coco['categories']}
ann_by_img = {}
for a in coco['annotations']:
    ann_by_img.setdefault(a['image_id'], []).append(a)

random.seed(42)
samples = random.sample(coco['images'], k=5)

fig, axes = plt.subplots(5, 1, figsize=(12, 30))
for ax, img_info in zip(axes, samples):
    img = Image.open(VAL_IMAGES / img_info['file_name'])
    ax.imshow(img)
    for ann in ann_by_img.get(img_info['id'], []):
        x, y, w, h = ann['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=1,
                                       edgecolor='lime', facecolor='none'))
        ax.text(x, max(0, y - 2), id_to_name[ann['category_id']],
                color='lime', fontsize=6)
    ax.set_title(f"{img_info['file_name']} - {len(ann_by_img.get(img_info['id'], []))} objs")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Zelle 5 - Groessenverteilungs-Histogramm (COCO-Buckets)
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('/content/visdrone')
splits = {
    'train/unsliced': ROOT / 'annotations' / 'train_unsliced.json',
    'val/unsliced':   ROOT / 'annotations' / 'val_unsliced.json',
    'train/sliced':   ROOT / 'annotations' / 'train_sliced.json',
    'val/sliced':     ROOT / 'annotations' / 'val_sliced.json',
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (label, path) in zip(axes.flat, splits.items()):
    coco = json.loads(path.read_text())
    areas = np.array([a['area'] for a in coco['annotations']])
    sides = np.sqrt(areas)
    ax.hist(sides, bins=60, color='steelblue', edgecolor='white')
    ax.axvline(32, color='orange', ls='--', label='small/medium (32)')
    ax.axvline(96, color='red', ls='--', label='medium/large (96)')
    ax.set_title(f'{label} - n={len(areas)}')
    ax.set_xlabel('sqrt(area) [px]')
    ax.set_ylabel('#annotations')
    ax.set_xlim(0, 300)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Zelle 6 - docs/data_report.md generieren (Skript ist single source of truth)
!PYTHONPATH=. python scripts/generate_data_report.py --visdrone /content/visdrone